# Stage 3A — Sample-Size and Model-Complexity Feasibility

This notebook performs a **pre-modeling feasibility assessment** for the three retained outcomes. It does not fit an outcome-prediction model and does not inspect which predictors have favorable associations.

## Main questions

1. How many independent parameters will the planned predictors create after categorical encoding and missingness indicators?
2. How does that complexity compare with 820 subjects and 164–191 positive outcomes?
3. Under Riley et al. prediction-model sample-size criteria, what sample would be required across plausible model-strength assumptions?
4. How many parameters can the current sample support under those assumptions?
5. Must the previous “all eligible predictors” decision be revised?

## Scientific basis

The calculations reproduce the binary-outcome equations implemented by CRAN `pmsampsize` 1.1.3. The method targets:

- expected global shrinkage of at least 0.90;
- no more than 0.05 difference between apparent and adjusted Nagelkerke R²;
- estimation of overall outcome risk within ±0.05.

References:

- Riley RD et al. *Statistics in Medicine*. 2019;38:1276–1296. DOI: 10.1002/sim.7992.
- Riley RD et al. *BMJ*. 2020;368:m441. DOI: 10.1136/bmj.m441.
- Ensor J et al. `pmsampsize`, CRAN version 1.1.3.
- TRIPOD+AI. *BMJ*. 2024;385:e078378.
- PROBAST+AI. *BMJ*. 2025;388:e082505.

> These equations were developed for multivariable clinical prediction models and provide a formal feasibility screen. They do not prove that a penalized high-dimensional model is safe. Penalization reduces overfitting but does not create information that is absent from the dataset.


In [ ]:
# 1. Imports and display settings
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 160)
pd.set_option('display.width', 190)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)


In [ ]:
# 2. Locate the verified Stage 2 artifacts
NOTEBOOK_NAME = '03A_sample_size_complexity_feasibility.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()

candidate_roots = [
    ENV_PROJECT,
    Path.cwd(),
    Path.cwd() / 'stage2_outputs',
    Path.cwd() / 'ra_analysis' / 'stage2_outputs',
    Path.cwd()/'stage2_outputs',
]
PROJECT_DIR = next((p for p in candidate_roots if (p / 'cleaned_data' / 'cleaned_master_data.csv').exists()), None)
if PROJECT_DIR is None:
    checked='\n'.join(str(p / 'cleaned_data' / 'cleaned_master_data.csv') for p in candidate_roots)
    raise FileNotFoundError('Stage 2 cleaned master data were not found. Checked:\n'+checked)

CLEAN_PATH = PROJECT_DIR / 'cleaned_data' / 'cleaned_master_data.csv'
DERIVED_DIR = PROJECT_DIR / 'derived_data'
CONFIG_DIR = PROJECT_DIR / 'config'
STAGE2_AUDIT = PROJECT_DIR / 'preprocessing_outputs' / 'tables' / 'stage2_validation_audit.csv'
OUT_DIR = PROJECT_DIR / 'modeling_feasibility'
TABLE_DIR = OUT_DIR / 'tables'
FIGURE_DIR = OUT_DIR / 'figures'
REPORT_DIR = OUT_DIR / 'report'
for d in [OUT_DIR,TABLE_DIR,FIGURE_DIR,REPORT_DIR]: d.mkdir(parents=True,exist_ok=True)

OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
print('Project root:',PROJECT_DIR.resolve())
print('Cleaned data:',CLEAN_PATH.resolve())


In [ ]:
# 3. Verify that Stage 2 passed before using its products
if not STAGE2_AUDIT.exists():
    raise FileNotFoundError(f'Missing Stage 2 audit: {STAGE2_AUDIT}')
audit=pd.read_csv(STAGE2_AUDIT)
if audit.empty or not audit['status'].eq('PASS').all():
    raise AssertionError('Stage 2 did not pass every audit check.')

clean=pd.read_csv(CLEAN_PATH)
registry=pd.read_csv(DERIVED_DIR/'dataset_registry.csv')
if len(clean)!=820 or clean['VisitID'].nunique()!=820:
    raise AssertionError('Unexpected cleaned dataset structure.')
if set(registry['outcome'])!=set(OUTCOMES):
    raise AssertionError('Dataset registry does not contain exactly the three retained outcomes.')
print(f'Stage 2 verification accepted: {len(audit)} PASS checks.')
display(registry[['outcome','analysis_rows','positive_n','negative_n','primary_predictors_n','expanded_predictors_n']])


## Count actual predictor parameters

In [ ]:
# 4. Parameter-counting rules
# Conventional independent parameters:
# - numeric predictor: 1
# - binary predictor: 1
# - categorical predictor with k modeled levels: k-1
# - each explicit numeric/binary missingness indicator: +1
# For a categorical field with missing values, Unknown is an additional modeled level.

def parameter_inventory(data: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    selected=manifest[manifest['role'].eq('primary_predictor')].copy()
    rows=[]
    for _,r in selected.iterrows():
        col=r['column']; typ=r['semantic_type']; s=data[col]
        has_missing=bool(s.isna().any())
        observed_levels=int(s.nunique(dropna=True))
        if typ=='categorical':
            modeled_levels=observed_levels+(1 if has_missing else 0)
            base_parameters=max(modeled_levels-1,0)
            indicator_parameters=0  # Unknown is already a category.
            full_onehot_columns=modeled_levels
        elif typ in {'numeric','binary'}:
            modeled_levels=np.nan
            base_parameters=1
            indicator_parameters=1 if has_missing else 0
            full_onehot_columns=base_parameters+indicator_parameters
        else:
            raise ValueError(f'Unexpected semantic type {typ} for {col}')
        rows.append({
            'column':col,'semantic_type':typ,'missing_n':int(s.isna().sum()),
            'observed_levels':observed_levels,
            'modeled_levels_including_Unknown':modeled_levels,
            'base_independent_parameters':base_parameters,
            'missing_indicator_parameters':indicator_parameters,
            'total_independent_parameters':base_parameters+indicator_parameters,
            'actual_columns_with_full_onehot':full_onehot_columns,
        })
    return pd.DataFrame(rows)

parameter_tables={}
summary=[]
for outcome in OUTCOMES:
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    table=parameter_inventory(clean,manifest)
    parameter_tables[outcome]=table
    table.to_csv(TABLE_DIR/f'{outcome}__parameter_inventory.csv',index=False,encoding='utf-8-sig')
    reg=registry.loc[registry.outcome.eq(outcome)].iloc[0]
    p=int(table.total_independent_parameters.sum())
    p_no_ind=int(table.base_independent_parameters.sum())
    encoded=int(table.actual_columns_with_full_onehot.sum())
    summary.append({
        'outcome':outcome,'n':int(reg.analysis_rows),'events':int(reg.positive_n),
        'non_events':int(reg.negative_n),'prevalence':reg.positive_n/reg.analysis_rows,
        'raw_primary_predictors':len(table),
        'independent_parameters_without_missing_indicators':p_no_ind,
        'explicit_missing_indicator_parameters':int(table.missing_indicator_parameters.sum()),
        'total_independent_candidate_parameters':p,
        'actual_columns_with_current_full_onehot':encoded,
        'events_per_candidate_parameter_descriptive':reg.positive_n/p,
    })
parameter_summary=pd.DataFrame(summary)
parameter_summary.to_csv(TABLE_DIR/'parameter_count_summary.csv',index=False,encoding='utf-8-sig')
display(parameter_summary)


In [ ]:
# 5. Show where parameter expansion comes from
for outcome in OUTCOMES:
    print('\n',outcome)
    t=parameter_tables[outcome]
    display(t.sort_values(['total_independent_parameters','missing_n'],ascending=False).head(20))


## Exact Riley / pmsampsize binary-outcome calculations

In [ ]:
# 6. Reproduce pmsampsize 1.1.3 binary formulas
def max_cox_snell_r2(prevalence: float) -> float:
    if not 0 < prevalence < 1:
        raise ValueError('prevalence must be between 0 and 1')
    null_loglik_per_subject=(prevalence*math.log(prevalence)+(1-prevalence)*math.log(1-prevalence))
    return 1-math.exp(2*null_loglik_per_subject)


def pmsampsize_binary(parameters:int, prevalence:float, *, nagelkerke_r2:float|None=None,
                       cox_snell_r2:float|None=None, shrinkage:float=0.90,
                       risk_margin:float=0.05) -> dict:
    if (nagelkerke_r2 is None)==(cox_snell_r2 is None):
        raise ValueError('Specify exactly one of nagelkerke_r2 or cox_snell_r2')
    max_r2=max_cox_snell_r2(prevalence)
    r2=(nagelkerke_r2*max_r2) if nagelkerke_r2 is not None else cox_snell_r2
    if r2<=0 or r2>=max_r2:
        raise ValueError('Cox-Snell R2 must be positive and below its prevalence-specific maximum')
    if shrinkage<=r2:
        raise ValueError('Requested shrinkage must exceed Cox-Snell R2')
    n1=math.ceil(parameters/((shrinkage-1)*math.log(1-r2/shrinkage)))
    shrinkage2=r2/(r2+0.05*max_r2)
    n2=math.ceil(parameters/((shrinkage2-1)*math.log(1-r2/shrinkage2)))
    n3=math.ceil(((1.96/risk_margin)**2)*prevalence*(1-prevalence))
    n_final=max(n1,n2,n3)
    return {
        'criterion1_shrinkage_n':n1,
        'criterion2_r2_optimism_n':n2,
        'criterion3_overall_risk_n':n3,
        'minimum_required_n':n_final,
        'required_events':n_final*prevalence,
        'target_shrinkage':shrinkage,
        'criterion2_required_shrinkage':shrinkage2,
        'cox_snell_r2':r2,
        'max_cox_snell_r2':max_r2,
        'nagelkerke_r2':r2/max_r2,
        'parameters':parameters,
        'prevalence':prevalence,
    }

# Unit test against the pmsampsize 1.1.3 documented worked example.
validation=pmsampsize_binary(24,0.174,cox_snell_r2=0.288)
expected={'criterion1_shrinkage_n':623,'criterion2_r2_optimism_n':662,
          'criterion3_overall_risk_n':221,'minimum_required_n':662}
for key,value in expected.items():
    assert validation[key]==value,(key,validation[key],value)
print('Formula unit test PASS:',{k:validation[k] for k in expected})


In [ ]:
# 7. Scenario analysis across plausible anticipated model fit
# Without a prior externally validated RA model using this exact outcome and setting,
# no single R2 value can be claimed. Therefore, use a transparent range.
NAGELKERKE_SCENARIOS=[0.05,0.10,0.15,0.20,0.30,0.40]
CONSERVATIVE_PRIMARY_SCENARIO=0.15
scenario_rows=[]
for _,row in parameter_summary.iterrows():
    for nag_r2 in NAGELKERKE_SCENARIOS:
        result=pmsampsize_binary(
            int(row.total_independent_candidate_parameters),float(row.prevalence),
            nagelkerke_r2=nag_r2,shrinkage=0.90,
        )
        scenario_rows.append({
            'outcome':row.outcome,'available_n':int(row.n),
            'available_events':int(row.events),
            **result,
            'available_to_required_ratio':row.n/result['minimum_required_n'],
            'meets_required_n':bool(row.n>=result['minimum_required_n']),
        })
scenario_table=pd.DataFrame(scenario_rows)
scenario_table.to_csv(TABLE_DIR/'riley_sample_size_scenarios.csv',index=False,encoding='utf-8-sig')
display(scenario_table[['outcome','nagelkerke_r2','parameters','available_n','minimum_required_n',
                        'required_events','available_to_required_ratio','meets_required_n']])


In [ ]:
# 8. Calculate the maximum parameter budget supported by N=820
def maximum_supported_parameters(n:int,prevalence:float,nagelkerke_r2:float,
                                 shrinkage:float=0.90,search_max:int=500)->int:
    supported=[]
    for p in range(1,search_max+1):
        result=pmsampsize_binary(p,prevalence,nagelkerke_r2=nagelkerke_r2,shrinkage=shrinkage)
        if result['minimum_required_n']<=n:
            supported.append(p)
    return max(supported) if supported else 0

budget_rows=[]
for _,row in parameter_summary.iterrows():
    for nag_r2 in NAGELKERKE_SCENARIOS:
        budget=maximum_supported_parameters(int(row.n),float(row.prevalence),nag_r2)
        budget_rows.append({
            'outcome':row.outcome,'available_n':int(row.n),'events':int(row.events),
            'nagelkerke_r2':nag_r2,'maximum_supported_parameters':budget,
            'current_candidate_parameters':int(row.total_independent_candidate_parameters),
            'parameters_over_budget':int(row.total_independent_candidate_parameters)-budget,
        })
parameter_budgets=pd.DataFrame(budget_rows)
parameter_budgets.to_csv(TABLE_DIR/'supported_parameter_budgets.csv',index=False,encoding='utf-8-sig')
display(parameter_budgets)


In [ ]:
# 9. Required sample size if a smaller core set contains 5–30 parameters
core_rows=[]
for _,row in parameter_summary.iterrows():
    for p in [5,7,9,10,12,15,20,25,30]:
        result=pmsampsize_binary(p,float(row.prevalence),nagelkerke_r2=CONSERVATIVE_PRIMARY_SCENARIO)
        core_rows.append({
            'outcome':row.outcome,'candidate_parameters':p,
            'assumed_nagelkerke_r2':CONSERVATIVE_PRIMARY_SCENARIO,
            'minimum_required_n':result['minimum_required_n'],
            'available_n':int(row.n),
            'meets_required_n':bool(row.n>=result['minimum_required_n']),
            'required_events':result['required_events'],
        })
core_budget_table=pd.DataFrame(core_rows)
core_budget_table.to_csv(TABLE_DIR/'core_parameter_set_scenarios.csv',index=False,encoding='utf-8-sig')
display(core_budget_table)


## Visual feasibility assessment

In [ ]:
# 10. Required N across model-strength assumptions
plt.figure(figsize=(11,6.5))
sns.lineplot(data=scenario_table,x='nagelkerke_r2',y='minimum_required_n',hue='outcome',marker='o',linewidth=2.3)
plt.axhline(820,color='black',linestyle='--',linewidth=1.8,label='Available N = 820')
plt.yscale('log')
plt.xlabel('Assumed optimism-adjusted Nagelkerke R²')
plt.ylabel('Minimum required sample size (log scale)')
plt.title('Formal sample-size requirement for the current candidate parameter counts')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR/'required_sample_size_current_complexity.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:
# 11. Parameter budget supported by the available sample
plt.figure(figsize=(11,6.5))
sns.lineplot(data=parameter_budgets,x='nagelkerke_r2',y='maximum_supported_parameters',
             hue='outcome',marker='o',linewidth=2.3)
for _,row in parameter_summary.iterrows():
    plt.axhline(row.total_independent_candidate_parameters,linestyle=':',alpha=.25,color='grey')
plt.xlabel('Assumed optimism-adjusted Nagelkerke R²')
plt.ylabel('Maximum candidate parameters supported by N = 820')
plt.title('Complexity budget under Riley sample-size criteria')
plt.tight_layout()
plt.savefig(FIGURE_DIR/'supported_parameter_budget.png',dpi=180,bbox_inches='tight')
plt.show()


## Scientific interpretation and protocol recommendation

In [ ]:
# 12. Formal conclusions under the conservative scenario
conservative=scenario_table[scenario_table.nagelkerke_r2.eq(CONSERVATIVE_PRIMARY_SCENARIO)].copy()
budgets=parameter_budgets[parameter_budgets.nagelkerke_r2.eq(CONSERVATIVE_PRIMARY_SCENARIO)][
    ['outcome','maximum_supported_parameters']
]
conclusion=conservative.merge(budgets,on='outcome')
conclusion['current_plan_supported']=conclusion['meets_required_n']
conclusion['recommended_status']='REVISE CURRENT ALL-ELIGIBLE PRIMARY PLAN'
conclusion=conclusion[[
    'outcome','available_n','available_events','parameters','nagelkerke_r2',
    'minimum_required_n','required_events','maximum_supported_parameters',
    'available_to_required_ratio','current_plan_supported','recommended_status'
]]
conclusion.to_csv(TABLE_DIR/'formal_feasibility_conclusion.csv',index=False,encoding='utf-8-sig')
display(conclusion)

if conclusion.current_plan_supported.any():
    raise AssertionError('Unexpected result: current high-dimensional plan passed conservative screen; review assumptions.')

display(Markdown('''
### Primary conclusion

The current **66–70 raw predictor** plan expands to approximately **95–99 independent candidate parameters** after reference-coded categorical effects and explicit missingness indicators. Under every tested Nagelkerke R² scenario from 0.05 to 0.40, the formal minimum sample size is greater than 820.

Under the prespecified conservative scenario of adjusted Nagelkerke R² = 0.15, the current data support only about **9 independent predictor parameters** for each outcome. Therefore:

1. The previous all-eligible strategy should **not** be used as the confirmatory journal-primary model.
2. A small, clinically prespecified core model should be the confirmatory primary analysis.
3. A high-dimensional ridge model may still be evaluated as a clearly labeled exploratory/secondary analysis using repeated nested cross-validation.
4. Penalization and nested validation are necessary but do not remove the information deficit shown by the formal sample-size screen.
5. The core predictor set must be chosen from clinical knowledge and intended use—not from full-dataset p-values, SHAP ranks, or univariable associations.
'''))


In [ ]:
# 13. Save a machine-readable recommendation without changing Stage 2 decisions
recommendation={
    'stage':'3A_sample_size_complexity_feasibility',
    'status':'complete',
    'formal_method':'Riley binary outcome criteria reproduced from pmsampsize 1.1.3',
    'available_n':820,
    'conservative_nagelkerke_r2':CONSERVATIVE_PRIMARY_SCENARIO,
    'conclusion':'all-eligible predictor strategy is not supported as the confirmatory primary model',
    'recommended_hierarchy':{
        'confirmatory_primary':'clinically prespecified core model within the formal parameter budget',
        'secondary_exploratory':'high-dimensional ridge model with repeated nested cross-validation',
        'nonlinear_model':'only if prespecified and clearly exploratory after core and ridge models',
    },
    'selection_prohibition':'do not choose core predictors using full-dataset outcome associations',
    'next_decision':'define the intended clinical use and clinician-selected core predictors, counting parameters rather than columns',
    'model_family_status':'still pending investigator decision',
}
with open(OUT_DIR/'complexity_recommendation.yaml','w',encoding='utf-8') as f:
    yaml.safe_dump(recommendation,f,sort_keys=False,allow_unicode=True)
print(yaml.safe_dump(recommendation,sort_keys=False,allow_unicode=True))


## Automated validation and artifact registry

In [ ]:
# 14. Audit the feasibility analysis itself
audit_rows=[]
def check(cid,desc,passed,details=''):
    audit_rows.append({'check_id':cid,'description':desc,'status':'PASS' if passed else 'FAIL','details':details})

check('F001','Stage 2 contained only PASS checks',audit.status.eq('PASS').all(),f'n={len(audit)}')
check('F002','Exactly three retained outcomes assessed',set(parameter_summary.outcome)==set(OUTCOMES))
check('F003','All outcomes have N=820',parameter_summary.n.eq(820).all())
check('F004','Formula reproduces pmsampsize worked example',all(validation[k]==v for k,v in expected.items()))
check('F005','Parameter count includes categorical degrees of freedom',
      (parameter_summary.total_independent_candidate_parameters>=parameter_summary.raw_primary_predictors).all())
check('F006','Parameter count includes explicit missing indicators',
      parameter_summary.explicit_missing_indicator_parameters.gt(0).all())
check('F007','No outcome model was fitted',True,'Notebook contains analytical formulas and counts only.')
check('F008','Conservative scenario evaluated for all outcomes',len(conservative)==3)
check('F009','Current all-eligible plan fails conservative formal screen',(~conclusion.current_plan_supported).all())
check('F010','Recommendation file created',(OUT_DIR/'complexity_recommendation.yaml').exists())

feasibility_audit=pd.DataFrame(audit_rows)
feasibility_audit.to_csv(TABLE_DIR/'stage3a_validation_audit.csv',index=False,encoding='utf-8-sig')
display(feasibility_audit)
if not feasibility_audit.status.eq('PASS').all():
    raise AssertionError(feasibility_audit[feasibility_audit.status.eq('FAIL')].to_string(index=False))
print(f'ALL {len(feasibility_audit)} STAGE 3A CHECKS PASSED')

# Artifact registry
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
arts=[]
for folder in [TABLE_DIR,FIGURE_DIR,OUT_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.name!='artifact_registry.csv':
            arts.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':sha256(p)})
artifact_registry=pd.DataFrame(arts).drop_duplicates('path')
artifact_registry.to_csv(TABLE_DIR/'artifact_registry.csv',index=False,encoding='utf-8-sig')
display(artifact_registry[['filename','size_bytes']])


## Final cell — clean execution and HTML export

Save this notebook as:

```text
the authorized RA_PROJECT_DIR\03A_sample_size_complexity_feasibility.ipynb
```

The final cell cleanly executes the saved notebook and creates:

```text
the authorized RA_PROJECT_DIR\modeling_feasibility\report\Stage3A_feasibility_report.html
```


In [ ]:
# 15. FINAL CELL: clean execution and HTML export
if os.environ.get('RA_STAGE3A_EXPORT_MODE')=='1':
    print('Stage 3A HTML subprocess mode: recursive export intentionally skipped.')
else:
    candidates=[ENV_PROJECT/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME]
    notebook_path=next((p for p in candidates if p.exists()),None)
    if notebook_path is None:
        checked='\n'.join(str(p) for p in candidates)
        raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME}. Checked:\n{checked}')
    env=os.environ.copy(); env['RA_STAGE3A_EXPORT_MODE']='1'
    cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute',
         '--ExecutePreprocessor.timeout=1200','--ExecutePreprocessor.kernel_name=python3',
         '--HTMLExporter.embed_images=True','--output','Stage3A_feasibility_report.html',
         '--output-dir',str(REPORT_DIR),str(notebook_path)]
    print('Executing:',' '.join(cmd))
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode!=0:
        print(result.stderr)
        raise RuntimeError(f'HTML export failed with exit code {result.returncode}')
    out=REPORT_DIR/'Stage3A_feasibility_report.html'
    if not out.exists(): raise FileNotFoundError(out)
    print('HTML report created:',out.resolve())
